# NSA vs Full Attention sur Colab T4

Ce notebook lance une comparaison courte entre un petit LM GPT-2 like avec `FullAttention` et une NSA **pure PyTorch** via `native_sparse_attention.pytorch_reference.ReferenceNativeSparseAttention`.

Le run est pensé pour un Colab gratuit T4 :
- séquence courte (`256`),
- presets `quick`, `standard`, `representative`,
- sous-ensemble de WikiText-2,
- dashboard live avec `train loss`, `validation loss`, temps par step, temps écoulé et tokens/s.

Par défaut le notebook part maintenant sur `RUN_PRESET = "representative"` pour un run plus significatif. Si c'est trop lent, passe à `standard` ou `quick`.

In [ ]:
REPO_URL_HTTPS = "https://github.com/gaspardbd/NativeSparseAttention.git"
REPO_DIR = "NativeSparseAttention"
OUTPUT_DIR = "outputs/colab_live"

RUN_PRESET = "standard"

EXPERIMENT_OVERRIDES = {
    "compare_nsa_branches": True,
    # Exemples: augmenter encore la duree si besoin.
    # "max_train_chunks": 2048,
    # "epochs": 4,
}


In [ ]:
import importlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

subprocess.run(
    [os.sys.executable, "-m", "pip", "install", "-q", "datasets", "transformers", "matplotlib", "tqdm", "einops"],
    check=True,
)

ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
WORK = ROOT / REPO_DIR
OUT = WORK / OUTPUT_DIR

if (WORK / ".git").exists():
    print("Repo deja clone -> git pull")
    subprocess.run(["git", "-C", str(WORK), "pull"], check=True)
else:
    if WORK.exists():
        shutil.rmtree(WORK)
    print("Clonage du repo")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL_HTTPS, str(WORK)], check=True)

OUT.mkdir(parents=True, exist_ok=True)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))

print("WORK =", WORK)
print("OUT  =", OUT)
subprocess.run(["git", "-C", str(WORK), "log", "-1", "--oneline"], check=True)


In [ ]:
import math

import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, clear_output, display

import train_nsa_vs_full
importlib.reload(train_nsa_vs_full)

display(Markdown("### Presets disponibles"))
print(json.dumps(train_nsa_vs_full.EXPERIMENT_PRESETS, indent=2))

cfg = train_nsa_vs_full.make_preset_config(RUN_PRESET, **EXPERIMENT_OVERRIDES)
model_specs = train_nsa_vs_full.get_model_specs(cfg)
model_labels = [(spec["progress_label"], spec["label"]) for spec in model_specs]
device = train_nsa_vs_full.get_device(cfg.device)
device_info = train_nsa_vs_full.get_device_info(device)
display(Markdown("### GPU / runtime"))
print(json.dumps(device_info, indent=2))

display(Markdown("### Configuration du run"))
print(cfg)


In [ ]:
class LiveDashboard:
    def __init__(self, smoothing_window):
        self.smoothing_window = smoothing_window
        self.events = {}

    def __call__(self, event):
        self.events[event["label"]] = event
        self.render()

    def render(self):
        clear_output(wait=True)
        lines = ["### Entrainement en cours"]
        for raw_label, pretty_label in model_labels:
            event = self.events.get(raw_label)
            if event is None:
                lines.append(f"- `{pretty_label}`: en attente")
                continue
            train_loss = "n/a" if math.isnan(event["train_loss"]) else f"{event['train_loss']:.3f}"
            eta_text = "n/a" if event["step"] == 0 else train_nsa_vs_full.format_duration(event["eta_sec"])
            gpu = event.get("runtime_stats", {})
            gpu_parts = []
            if "torch_mem_allocated_gb" in gpu:
                gpu_parts.append(f"alloc {gpu['torch_mem_allocated_gb']:.2f}G")
            if "torch_mem_reserved_gb" in gpu:
                gpu_parts.append(f"reserved {gpu['torch_mem_reserved_gb']:.2f}G")
            if "gpu_util_pct" in gpu:
                gpu_parts.append(f"util {gpu['gpu_util_pct']:.0f}%")
            gpu_text = " | " + " | ".join(gpu_parts) if gpu_parts else ""
            lines.append(
                f"- `{pretty_label}`: step {event['step']}/{event['total_steps']} | "
                f"train {train_loss} | val {event['val_loss']:.3f} | "
                f"elapsed {train_nsa_vs_full.format_duration(event['elapsed_sec'])} | eta {eta_text} | "
                f"step {event['avg_step_time_sec']:.2f}s | "
                f"{event['tokens_per_sec']:.0f} tok/s | eval {event['eval_time_sec']:.2f}s{gpu_text}"
            )
        display(Markdown("\n".join(lines)))

        ncols = 3 if cfg.show_elapsed_time_plot else 2
        fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 4))
        if ncols == 1:
            axes = [axes]
        for raw_label, pretty_label in model_labels:
            event = self.events.get(raw_label)
            if event is None:
                continue
            history = event["history"]
            smooth = train_nsa_vs_full.smooth_curve(history["train_loss"], self.smoothing_window)
            axes[0].plot(history["train_steps"], smooth, label=pretty_label)
            axes[1].plot(
                history["val_steps"],
                history["val_loss"],
                label=pretty_label,
                **train_nsa_vs_full.validation_plot_kwargs(len(history["val_steps"])),
            )
            if cfg.show_elapsed_time_plot:
                axes[2].plot(
                    [elapsed / 60.0 for elapsed in history["val_elapsed_sec"]],
                    history["val_loss"],
                    label=pretty_label,
                    **train_nsa_vs_full.validation_plot_kwargs(len(history["val_elapsed_sec"])),
                )

        axes[0].set_title("Training Loss")
        axes[0].set_xlabel("Step")
        axes[0].set_ylabel("Loss")
        axes[0].grid(True, alpha=0.3)
        axes[0].legend()

        axes[1].set_title("Validation Loss vs Step")
        axes[1].set_xlabel("Step")
        axes[1].set_ylabel("Validation Loss")
        axes[1].grid(True, alpha=0.3)
        axes[1].legend()

        if cfg.show_elapsed_time_plot:
            axes[2].set_title("Validation Loss vs Elapsed Time")
            axes[2].set_xlabel("Elapsed time (minutes)")
            axes[2].set_ylabel("Validation Loss")
            axes[2].grid(True, alpha=0.3)
            axes[2].legend()

        plt.tight_layout()
        display(fig)
        plt.close(fig)


dashboard = LiveDashboard(cfg.smoothing_window)


In [ ]:
results = train_nsa_vs_full.run_experiment(
    cfg=cfg,
    output_dir=OUT,
    progress_callback=dashboard,
    show_progress_bar=False,
)

dashboard.render()
display(Markdown("### Resume final"))
print(json.dumps(results["summary"], indent=2))
print("Artifacts sauves dans", OUT)


In [ ]:
fig_path = OUT / "nsa_vs_full.png"
json_path = OUT / "results.json"

if fig_path.is_file():
    display(Image(filename=str(fig_path)))
else:
    print("Figure absente:", fig_path)

print("JSON:", json_path)


In [ ]:
# Optionnel: copie vers Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_out = Path("/content/drive/MyDrive/NSA_experiments")
    drive_out.mkdir(parents=True, exist_ok=True)
    for file_path in OUT.glob("*"):
        shutil.copy2(file_path, drive_out / file_path.name)
    print("Copie vers", drive_out)
except ImportError:
    print("Pas sur Colab -> ignore")
